# Лабораторная работа 6. DTW и классификация рядов

**Цель.** Показать, как динамическая деформация времени помогает сравнивать временные ряды разной формы.


## Ход работы

Сначала генерируются несколько классов синтетических сигналов, затем расстояние между рядами считается вручную и через библиотеку. В конце эти расстояния используются в классификации.


## Подготовка

Задаю seed и подключаю библиотеки. Синтетические ряды создаются случайно, поэтому фиксированное состояние нужно для повторяемого набора сигналов.


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Типы сигналов

Функции задают простые формы: импульс, гармонический сигнал и другие варианты. Дальше из них собирается набор рядов, похожий на маленький датасет для классификации.


In [ ]:
def pulse(t):
    """Пульс сигнал"""
    return 1 * (abs(t) < 0.5)

In [ ]:
# индекс и отсчет времени в секундах
time_index = np.linspace(0, 9, 100)

tseries_list = {'Time': time_index}
d = np.random.random(size=10)


N = 7 #количество образцов в каждом типе сигналов


# гармонические колебания
f0 = 0.2

for i in range(N):
    # Добавляем небольшую случайную фазу и амплитуду для разнообразия
    phase = d[i % len(d)] * np.pi
    amp = 1 + 0.1 * (i - 3)
    tseries_list["Tc"+str(i)] = amp * np.cos(2 * np.pi * f0 * time_index + phase)

# модифицированный синус
for i in range(N):
    # Модифицированный синус: синус с измененной частотой
    freq_mod = f0 * (1 + 0.1 * i)
    tseries_list["Ts"+str(i)] = np.sin(2 * np.pi * freq_mod * time_index)

# пульс сигнал
for i in range(N):
    # Пульс-сигнал: последовательность импульсов
    period = 2.0 + 0.1 * i # меняем период
    signal = np.zeros_like(time_index)
    for t_idx, t_val in enumerate(time_index):
        # Создаем импульсы каждые period секунд
        if abs((t_val % period) - period/2) < 0.1: # узкий импульс
            signal[t_idx] = 1.0
    tseries_list["Tp"+str(i)] = signal


# отрисовка всех сигналов
plt.figure(figsize=(15, 8))

# Ts - синие
for i in range(N):
    plt.plot(time_index, tseries_list["Ts"+str(i)], 'b-', alpha=0.6, label='Ts' if i == 0 else "")

# Tc - зеленые
for i in range(N):
    plt.plot(time_index, tseries_list["Tc"+str(i)], 'g-', alpha=0.6, label='Tc' if i == 0 else "")

# Tp - красные
for i in range(N):
    plt.plot(time_index, tseries_list["Tp"+str(i)], 'r-', alpha=0.6, label='Tp' if i == 0 else "")

plt.title(r'Временные ряды')
plt.xlabel(r't (in s)')
plt.ylabel('Amplitude')
plt.legend()
plt.grid()
plt.show()

## Матрица расстояний и DTW

Сначала реализую базовую матрицу расстояний и накопленную матрицу DTW вручную. Это полезно, потому что становится видно, откуда берётся итоговое расстояние между двумя рядами.


In [ ]:
def distance_matrix(x, y, q) -> np.array:
    """
    Функция  рассчета  матрицы  расстояний между точками двух рядов
    """
    mdist = np.zeros((len(y), len(x)))
    for i in range(len(y)):
        for j in range(len(x)):
            mdist[i, j] = abs(x[j] - y[i]) ** q

    return mdist

In [ ]:
def DTW(x, x_s, q=1, isDTW=True):
    '''
        x: первый ряд
        x_s : второй ряд
        q : степень для вычисления базового расстояния
    '''
    N = len(x) # Считаем, что ряды равной длины. Но это не всегда так, перепишите функцию для
               # вычисления расстояния между рядами, длины которых не равны

    # Строим матрицу согласно реккурентной формуле, полученной выше. Матрица в нашем случае будет размерности

    dist = distance_matrix(x,x_s,q=2)

    R = [[0] * (N+1) for i in range(N+1)]

    # Исправление границ: нужно заполнить до N+1, чтобы захватить весь крайний ряд/столбец
    for i in range(1, N+1):
        R[i][0] = dist[i-1,0] + R[i-1][0] # dist индексируется с 0, R с 1 (из-за padding)
        R[0][i] = dist[0,i-1] + R[0][i-1]

    k = 0
    if isDTW:
        k = 1

    # начинаем расчет по реккурентной формуле

    for i in range(1, N+1):
        for j in range(1, N+1):
            # R[i][j] = dist[i-1][j-1] + min(R[i-1][j], R[i][j-1], R[i-1][j-1])
            R[i][j] = dist[i-1, j-1] + min(R[i-1][j], R[i][j-1], R[i-1][j-1])

    # ищем минимальный путь. Начинаем от нижнего правого угла.
    pth = []   # лист с кортежами индексов пути
    i = N
    j = N
    while i >= 0 and j >= 0:
        pth.append((i,j))
        if i == 0 and j == 0:
            break

        val_diag = R[i-1][j-1] if i > 0 and j > 0 else float('inf')
        val_up = R[i-1][j] if i > 0 else float('inf')
        val_left = R[i][j-1] if j > 0 else float('inf')

        min_val = min(val_diag, val_up, val_left)

        if min_val == val_left:
            I = i
            J = j - 1
        elif min_val == val_up:
            I = i - 1
            J = j
        else:
            I = i - 1
            J = j - 1

        i = I
        j = J


    #print("\nПуть:")
    #print(pth)

    #Считаем расстояние между двумя рядами
    s = 0
    for l in pth:
        # Важно: суммируем значения из матрицы накопленных стоимостей R вдоль пути
        s += R[l[0]][l[1]]

    # Нормализация (среднее значение вдоль пути)
    if len(pth) > 0:
        s = s / len(pth)
    else:
        s = 0

    return s, pth, R

## Сравнение выбранных рядов

На нескольких парах сигналов смотрю, как меняется путь выравнивания. Если формы похожи, DTW находит диагональный или почти диагональный маршрут; при сдвигах путь растягивается.


In [ ]:
# Посчитаем DTW для двух временных рядов

#для простоты выделяем ряды, которые будем сравнивать
x = np.abs(tseries_list.get("Ts1", np.zeros(100))) # Если tseries_list не заполнен
x_s = np.abs(tseries_list.get("Ts6", np.zeros(100)))
x_p = np.abs(tseries_list.get("Tp2", np.zeros(100)))

s1 = DTW(x, x_s, q=2)
s2 = DTW(x, x_p, q=2)

print (f"DTW\nTs1 и Ts6 = {s1[0]}     Ts1 и Tp2 = {s2[0]}")

s11 = DTW(x, x_s, q=2, isDTW=False)
s21 = DTW(x, x_p, q=2, isDTW=False)

print (f"Dec\nTs1 и Ts6 = {s11[0]}     Ts1 и Tp2 = {s21[0]}")

In [ ]:
import seaborn as sbn
# Отрисуем матрицы весов расстояний Ts1 и Ts3

cost_matrix = s1[2]
warp_path = s1[1]

fig, ax = plt.subplots(figsize=(12, 8))
ax = sbn.heatmap(cost_matrix, square=True, linewidths=0.1, cmap="YlGnBu", ax=ax)
ax.invert_yaxis()


path_x = [p[0] for p in warp_path]
path_y = [p[1] for p in warp_path]

# Align the path from the center of each cell
path_xx = [x+0.5 for x in path_x]
path_yy = [y+0.5 for y in path_y]

ax.plot(path_xx, path_yy, color='blue', linewidth=3, alpha=0.2)

In [ ]:
fig, ax = plt.subplots(figsize=(30, 15))


warp_path = s1[1]
x1 = x
x2 = x_s

# Remove the border and axes ticks
fig.patch.set_visible(True)
ax.axis('off')

for [map_x, map_y] in warp_path:
    ax.set_facecolor('white')
    ax.plot([map_x-1, map_y-1], [x1[map_x-1], x2[map_y-1]], '-k')

ax.plot(x1, color='blue', marker='o', markersize=10, linewidth=5)
ax.plot(x2, color='red', marker='o', markersize=10, linewidth=5)
ax.tick_params(axis="both", which="major", labelsize=18)

In [ ]:
# Отрисуем матрицы весов расстояний Ts1 и Tp1

cost_matrix = s2[2]
warp_path = s2[1]

fig, ax = plt.subplots(figsize=(12, 8))
ax = sbn.heatmap(cost_matrix, square=True, linewidths=0.1, cmap="YlGnBu", ax=ax)
ax.invert_yaxis()


path_x = [p[0] for p in warp_path]
path_y = [p[1] for p in warp_path]

# Align the path from the center of each cell
path_xx = [x+0.5 for x in path_x]
path_yy = [y+0.5 for y in path_y]

ax.plot(path_xx, path_yy, color='blue', linewidth=3, alpha=0.2)

In [ ]:
# Отрисуем матрицы весов расстояний Ts1 и Tp1 без DTW

cost_matrix = s21[2]
warp_path = s21[1]

fig, ax = plt.subplots(figsize=(12, 8))
ax = sbn.heatmap(cost_matrix, square=True, linewidths=0.1, cmap="YlGnBu", ax=ax)
ax.invert_yaxis()


path_x = [p[0] for p in warp_path]
path_y = [p[1] for p in warp_path]

# Align the path from the center of each cell
path_xx = [x+0.5 for x in path_x]
path_yy = [y+0.5 for y in path_y]

ax.plot(path_xx, path_yy, color='blue', linewidth=3, alpha=0.2)

## Библиотечная реализация

После ручной функции использую `dtaidistance`. Результат можно сравнить с собственной реализацией и убедиться, что логика выравнивания совпадает.


In [ ]:
from dtaidistance import dtw

x = np.abs(tseries_list["Ts1"])
x_s = np.abs(tseries_list["Ts6"])

# Вычислите расстояние и пути с помощью встроенной функции dtw.warping_paths из библиотеки dtaidistance
distance, paths = dtw.warping_paths(x, x_s)
print(distance)
print(paths)

In [ ]:
from dtaidistance import dtw
from dtaidistance import dtw_visualisation as dtwvis
import random
import numpy as np
xw = np.arange(0, 20, .5)
s1 = x
s2 = x_s
random.seed(1)
for idx in range(len(s2)):
    if random.random() < 0.05:
        s2[idx] += (random.random() - 0.5) / 2
d, paths = dtw.warping_paths(s1, s2, window=25, psi=2)
best_path = dtw.best_path(paths)
dtwvis.plot_warpingpaths(s1, s2, paths, path=best_path)

In [ ]:
x = np.abs(tseries_list["Ts1"])
x_p = np.abs(tseries_list["Tp2"])


distance, paths = dtw.warping_paths(x, x_p)
print(distance)
print(paths)

In [ ]:
from dtaidistance import dtw
from dtaidistance import dtw_visualisation as dtwvis
import random
import numpy as np
xw = np.arange(0, 20, .5)
s1 = x
s2 = x_p
random.seed(1)
for idx in range(len(s2)):
    if random.random() < 0.05:
        s2[idx] += (random.random() - 0.5) / 2
d, paths = dtw.warping_paths(s1, s2, window=25, psi=2)
best_path = dtw.best_path(paths)
dtwvis.plot_warpingpaths(s1, s2, paths, path=best_path)

## Подготовка выборки

Собираю массив признаков и метки классов из созданных временных рядов. Затем случайно делю объекты на обучение и тест, чтобы проверить классификаторы на новых примерах.


In [ ]:
# переделываем датасет так, чтобы с ним можно было бы работать

X = []  # значения
Y = []  # целевая переменная

for v in tseries_list:
    if v != 'Time':
        X.append(tseries_list[v])
        c = v[:-1]
        if c == "Ts":
            Y.append(0)
        elif c == "Tc":
            Y.append(1)
        else:
            Y.append(2)

arr = np.arange(len(Y))
np.random.shuffle(arr)


X_train = []
X_test = []
y_train = []
y_test = []

# Для этого отделите от перемешанных индексов arr последние 5 элементов под тест,
# а остальные возьмите для трейна.

# Индексы для теста: последние 5 элементов перемешанного массива
test_indices = arr[-5:]
# Индексы для трейна: все остальные элементы
train_indices = arr[:-5]

# Заполняем тестовые выборки
for idx in test_indices:
    X_test.append(np.array(X[idx])) # Преобразуем список в numpy array
    y_test.append(Y[idx])

# Заполняем обучающие выборки
for idx in train_indices:
    X_train.append(np.array(X[idx])) # Преобразуем список в numpy array
    y_train.append(Y[idx])

# Преобразуем списки в numpy массивы
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

## Классификация

Сравниваю KNN с DTW-метрикой, TimeSeriesForest и KNN с евклидовым расстоянием. Разница между ними показывает, насколько важен способ измерения близости рядов.


In [ ]:
from pyts.classification import KNeighborsClassifier

# Инициализируем классификатор с метрикой DTW
clf = KNeighborsClassifier(metric='dtw')

# Обучаем модель
clf.fit(X_train, y_train)

# Проверяем точность и делаем предсказания (раскомментируйте для проверки)
print(f"Accuracy {clf.score(X_test, y_test)}")
print(f"Вектор вероятности принадлежности к классам {clf.predict_proba(X_test[2].reshape(1, -1))}")
print(f"Истинный класс для предсказаний {y_test[2]}")

In [ ]:
import numpy as np
from pyts.classification import TimeSeriesForest
import matplotlib.pyplot as plt

# Инициализируем классификатор TimeSeriesForest с фиксированным random_state для воспроизводимости
clf = TimeSeriesForest(random_state=43)

# Обучаем модель
clf.fit(X_train, y_train)

# Проверяем точность и делаем предсказания (раскомментируйте для проверки)
print(f"Accuracy {clf.score(X_test, y_test)}")
print(f"Вектор вероятности принадлежности к классам {clf.predict_proba(X_test[2].reshape(1, -1))}")
print(f"Истинный класс для предсказаний {y_test[2]}")

In [ ]:
# В домашнем задании требовалось сделать 3 модели на выбор.

from pyts.classification import KNeighborsClassifier

# Инициализируем классификатор с евклидовой метрикой
clf_eucl = KNeighborsClassifier(metric='euclidean')

# Обучаем модель
clf_eucl.fit(X_train, y_train)

# Проверяем точность и делаем предсказания (раскомментируйте для проверки)
print(f"Accuracy {clf_eucl.score(X_test, y_test)}")
print(f"Вектор вероятности принадлежности к классам {clf_eucl.predict_proba(X_test[2].reshape(1, -1))}")
print(f"Истинный класс для предсказаний {y_test[2]}")

## Вывод

DTW лучше подходит для сигналов, которые похожи по форме, но сдвинуты или растянуты по времени. В классификации это может дать преимущество по сравнению с обычным евклидовым расстоянием.
